# Machine Learning: Mushroom Classification

**Author:** Roberto Jourdain

**Dataset:** Mushroom Classification — sourced from [Kaggle (UCI)](https://www.kaggle.com/datasets/uciml/mushroom-classification)

This project builds a supervised classification model to predict whether a mushroom is edible or poisonous based on its physical characteristics. The dataset contains 8,124 samples with 22 categorical features describing properties such as cap shape, odor, gill color, and habitat. The task is a binary classification problem using scikit-learn.

## Data Loading and Inspection

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [2]:
df = pd.read_csv("mushrooms.csv")
df.head()

,class,cap-shape,cap-surface,cap-color,bruises,odor,gill-attachment,gill-spacing,gill-size,gill-color,...,stalk-surface-below-ring,stalk-color-above-ring,stalk-color-below-ring,veil-type,veil-color,ring-number,ring-type,spore-print-color,population,habitat
0,p,x,s,n,t,p,f,c,n,k,...,s,w,w,p,w,o,p,k,s,u
1,e,x,s,y,t,a,f,c,b,k,...,s,w,w,p,w,o,p,n,n,g
2,e,b,s,w,t,l,f,c,b,n,...,s,w,w,p,w,o,p,n,n,m
3,p,x,y,w,t,p,f,c,n,n,...,s,w,w,p,w,o,p,k,s,u
4,e,x,s,g,f,n,f,w,b,k,...,s,w,w,p,w,o,e,n,a,g


In [3]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8124 entries, 0 to 8123
Data columns (total 23 columns):
 #   Column                    Non-Null Count  Dtype 
---  ------                    --------------  ----- 
 0   class                     8124 non-null   object
 1   cap-shape                 8124 non-null   object
 2   cap-surface               8124 non-null   object
 3   cap-color                 8124 non-null   object
 4   bruises                   8124 non-null   object
 5   odor                      8124 non-null   object
 6   gill-attachment           8124 non-null   object
 7   gill-spacing              8124 non-null   object
 8   gill-size                 8124 non-null   object
 9   gill-color                8124 non-null   object
 10  stalk-shape               8124 non-null   object
 11  stalk-root                8124 non-null   object
 12  stalk-surface-above-ring  8124 non-null   object
 13  stalk-surface-below-ring  8124 non-null   object
 14  stalk-color-above-ring  

In [4]:
df.isnull().sum().sum()

np.int64(0)

In [5]:
df["class"].value_counts()

class
e    4208
p    3916
Name: count, dtype: int64

**Inspection notes:** The dataset has 8,124 rows and 23 columns, all categorical (object dtype). There are no missing values. The target variable `class` has two values: `e` (edible) and `p` (poisonous). The classes are reasonably balanced, which means we do not need to apply oversampling or class weighting techniques.

## Data Preparation and Preprocessing

All 23 columns are categorical strings (single-letter codes). Machine learning models require numeric input, so we apply label encoding to convert each column's categories into integers. To avoid data leakage, we first split the raw data into training and testing sets, then fit a separate `LabelEncoder` on each column using only the training data. The fitted encoders are then used to transform the test data. This ensures the model never sees information from the test set during preprocessing.

In [ ]:
# Separate features and target before any encoding
X_raw = df.drop(columns=["class"])
y_raw = df["class"]

# Split raw data into training and testing sets first
X_train_raw, X_test_raw, y_train, y_test = train_test_split(
    X_raw, y_raw, test_size=0.2, random_state=42
)

# Encode the target variable
target_le = LabelEncoder()
y_train = target_le.fit_transform(y_train)
y_test = target_le.transform(y_test)

# Fit a LabelEncoder per feature column on training data only, then transform both sets
encoders = {}
X_train = pd.DataFrame(index=X_train_raw.index)
X_test = pd.DataFrame(index=X_test_raw.index)

for col in X_raw.columns:
    enc = LabelEncoder()
    X_train[col] = enc.fit_transform(X_train_raw[col])
    X_test[col] = enc.transform(X_test_raw[col])
    encoders[col] = enc

print(f"Training set: {X_train.shape[0]} samples")
print(f"Testing set:  {X_test.shape[0]} samples")
X_train.head()

## Model Selection and Training

We train two models to compare performance:

1. **Decision Tree** — a single tree that learns decision rules from the data. Simple and interpretable, but prone to overfitting.
2. **Random Forest** — an ensemble of 100 decision trees that combines their predictions through majority voting (Breiman, 2001). More robust than a single tree due to averaging, and provides feature importance scores.

Comparing both helps demonstrate whether the ensemble approach adds value over a simpler model on this dataset.

In [ ]:
# Train a Decision Tree classifier
dt_model = DecisionTreeClassifier(random_state=42)
dt_model.fit(X_train, y_train)
dt_pred = dt_model.predict(X_test)

# Train a Random Forest classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)
rf_model.fit(X_train, y_train)
rf_pred = rf_model.predict(X_test)

print("Decision Tree Accuracy:", f"{accuracy_score(y_test, dt_pred):.4f}")
print("Random Forest Accuracy:", f"{accuracy_score(y_test, rf_pred):.4f}")

## Model Evaluation

In [ ]:
# Full classification report for Random Forest
print("Random Forest Classification Report:\n")
print(classification_report(y_test, rf_pred, target_names=["edible", "poisonous"]))

In [ ]:
# Figure 1: Confusion Matrix (Random Forest)
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, rf_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["edible", "poisonous"],
            yticklabels=["edible", "poisonous"], ax=ax)
ax.set_title("Confusion Matrix (Random Forest)")
ax.set_xlabel("Predicted Label")
ax.set_ylabel("True Label")
plt.tight_layout()
plt.show()

In [ ]:
# Figure 2: Top 10 Feature Importances (Random Forest)
importances = pd.Series(rf_model.feature_importances_, index=X_train.columns)
top_features = importances.sort_values(ascending=False).head(10)

fig, ax = plt.subplots(figsize=(10, 5))
top_features.plot(kind="barh", color="steelblue", ax=ax)
ax.set_title("Top 10 Feature Importances (Random Forest)")
ax.set_xlabel("Importance Score")
ax.set_ylabel("Feature")
ax.invert_yaxis()
plt.tight_layout()
plt.show()

## Summary

This project built and compared two classifiers -- a Decision Tree and a Random Forest -- to predict whether a mushroom is edible or poisonous based on 22 physical characteristics. Both models achieved near-perfect accuracy on the test set, which highlights that the dataset is nearly perfectly separable rather than indicating exceptional model performance. The feature importance analysis confirmed that `odor` is by far the most predictive characteristic, consistent with mycological domain knowledge. A key methodological decision was fitting the label encoders on training data only and applying them to the test set separately, preventing data leakage during preprocessing. While the dataset is effective for demonstrating a complete ML workflow, the perfect results limit what can be learned about model tuning or handling difficult classification boundaries -- real-world mushroom identification involves continuous, noisy features and environmental factors not captured here, and this model should never be used as a standalone foraging guide.